# Avida CFO – contract extraction (runs on OpenShift AI Workbench)

This notebook builds the knowledge base the **Contract Agent** uses:

1. Render the demo contracts (`backend/data/contracts/*.md`) as PDFs (in real life the owner uploads PDFs).
2. Extract text page by page with `pypdf`.
3. Ask the **vLLM-served model on OpenShift AI** to extract key terms + clauses as structured JSON.
4. Benchmark the extraction against the hand-labelled ground truth.
5. Write `backend/data/contracts_index.json`.

Run it in a JupyterLab workbench in the same cluster (the internal service URL works there).

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json, os
from pathlib import Path
from openai import OpenAI

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = REPO / "backend" / "data"
CONTRACTS = DATA / "contracts"

LLM_BASE_URL = os.getenv("LLM_BASE_URL", "http://qwen25-7b-instruct-predictor.avida-models.svc.cluster.local:8080/v1")
LLM_MODEL = os.getenv("LLM_MODEL", "qwen25-7b-instruct")
client = OpenAI(base_url=LLM_BASE_URL, api_key=os.getenv("LLM_API_KEY", "EMPTY"))
print([m.id for m in client.models.list().data])

## 1. Render the demo contracts as PDF

In [ ]:
from fpdf import FPDF

def md_to_pdf(md_path: Path, pdf_path: Path):
    pdf = FPDF()
    pdf.set_auto_page_break(True, margin=15)
    pdf.add_page()
    for line in md_path.read_text(encoding="utf-8").splitlines():
        text = line.replace("\u2013", "-").replace("**", "").lstrip("# ").strip()
        if line.startswith("## "):
            pdf.add_page()  # one section per page -> realistic page references
            pdf.set_font("Helvetica", "B", 13)
        elif line.startswith("# "):
            pdf.set_font("Helvetica", "B", 16)
        else:
            pdf.set_font("Helvetica", size=11)
        pdf.multi_cell(0, 7, text)
    pdf.output(str(pdf_path))

for md_file in CONTRACTS.glob("*.md"):
    md_to_pdf(md_file, md_file.with_suffix(".pdf"))
sorted(p.name for p in CONTRACTS.glob("*.pdf"))

## 2. Extract text per page

In [ ]:
from pypdf import PdfReader

def pdf_pages(path: Path) -> list[str]:
    return [p.extract_text() or "" for p in PdfReader(str(path)).pages]

docs = {p.name: pdf_pages(p) for p in CONTRACTS.glob("*.pdf")}
{k: len(v) for k, v in docs.items()}

## 3. LLM extraction with a JSON schema (vLLM guided decoding)

In [ ]:
SCHEMA = {
    "type": "object",
    "properties": {
        "supplier": {"type": "string"},
        "title": {"type": "string"},
        "start_date": {"type": "string"},
        "monthly_fee": {"type": "number"},
        "fixed_price_until": {"type": ["string", "null"]},
        "price_change_notice_days": {"type": ["integer", "null"]},
        "max_annual_increase_pct": {"type": ["number", "null"]},
        "clauses": {"type": "array", "items": {"type": "object", "properties": {
            "id": {"type": "string"}, "page": {"type": "integer"}, "title": {"type": "string"}, "text": {"type": "string"}},
            "required": ["id", "page", "title", "text"]}},
    },
    "required": ["supplier", "title", "start_date", "monthly_fee", "fixed_price_until",
                 "price_change_notice_days", "max_annual_increase_pct", "clauses"],
}
SYSTEM = (
    "You extract key commercial terms from a supplier contract for a small company. "
    "Dates as YYYY-MM-DD. fixed_price_until = last day of any price guarantee, else null. "
    "Notice periods in days (3 months = 90). Only include clauses about fees, prices, indexation, notice or term; "
    "copy their text verbatim and give the page number. The supplier is the party that is not the Customer. JSON only."
)

def extract(pages: list[str]) -> dict:
    text = "\n\n".join(f"[page {i+1}]\n{t}" for i, t in enumerate(pages))
    r = client.chat.completions.create(
        model=LLM_MODEL, temperature=0,
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}],
        response_format={"type": "json_schema", "json_schema": {"name": "contract", "schema": SCHEMA}},
    )
    return json.loads(r.choices[0].message.content)

extracted = {name: extract(pages) for name, pages in docs.items()}
extracted

## 4. Benchmark against the hand-labelled index

In [ ]:
truth = {c["supplier"]: c for c in json.loads((DATA / "contracts_index.json").read_text(encoding="utf-8"))["contracts"]}
FIELDS = ["start_date", "monthly_fee", "fixed_price_until", "price_change_notice_days", "max_annual_increase_pct"]

def match_truth(supplier: str) -> dict:
    return next((v for k, v in truth.items() if k.split()[0].lower() in supplier.lower()), {})

rows, hits = [], 0
for name, ex in extracted.items():
    gt = match_truth(ex["supplier"])
    if not gt:
        print("no ground truth for", ex["supplier"]); continue
    for f in FIELDS:
        ok = ex.get(f) == gt.get(f)
        hits += ok
        rows.append((gt["supplier"], f, gt.get(f), ex.get(f), "OK" if ok else "MISMATCH"))
for r in rows:
    print(*r, sep=" | ")
print(f"\nField accuracy: {hits}/{len(rows)} = {hits/len(rows):.0%}")

## 5. Write the index used by the backend

Keeps ids/aliases from the existing file. Set `OVERWRITE = True` only if the benchmark looks good.

In [ ]:
OVERWRITE = False

out = []
for i, (name, ex) in enumerate(sorted(extracted.items()), start=1):
    gt = match_truth(ex["supplier"])
    out.append({
        "id": gt.get("id", f"C-{i:03d}"),
        "aliases": gt.get("aliases", [ex["supplier"].split()[0].lower()]),
        "file": f"contracts/{name}",
        **ex,
    })

target = DATA / ("contracts_index.json" if OVERWRITE else "contracts_index.generated.json")
target.write_text(json.dumps({"_comment": "Generated by notebooks/contract_extraction.ipynb", "contracts": out},
                             indent=2, ensure_ascii=False), encoding="utf-8")
print("written", target)